# SFT 데이터 만들기: 요청 → 이야기

**11장의 요청-이야기 데이터를 만든 노트북입니다.** 실행하지 않아도 책을 따라가는 데 문제가 없어요. 11장은 여기서 만든 데이터셋을 `load_sft_texts`로 불러옵니다. 같은 방법을 여러분의 데이터에 써 보고 싶을 때 참고하세요.

**하는 일:** Tiny-Ko-Stories의 이야기 앞부분에서 주인공 이름·동물·장소·때를 찾아, 그 이야기를 요청하는 문장을 자동으로 만듭니다.

```
요청: 숲속에서 토끼 보리가 나오는 이야기를 들려줘
이야기: 햇볕이 쨍쨍한 여름날, 꼬마 토끼 보리가 숲속 모험을 떠나요. ...
```

**흐름:** 설정 → 이야기 불러오기 → (선택) 이름 후보 확인 → 특징 찾기 규칙 → 쌍 만들기와 점검 → 글자 점검 → 나누기·저장 → 허깅페이스 업로드

### 원 데이터
[Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)). 가공본도 CC BY 4.0으로 공개하고 출처와 변경 사항을 데이터셋 카드에 적습니다.


In [ ]:
%pip install -q datasets huggingface_hub


## 1. 설정

- `SKIP`: 사전학습에 쓸 앞부분을 건너뛰어, 사전학습과 SFT가 **서로 다른 이야기**를 쓰게 합니다.
- `MAX_LEN`: 이보다 긴 이야기는 뺍니다. 요청 문장(약 30자)과 형식 글자를 합쳐 모델의 문맥 길이(384글자) 안에 들어가야 하기 때문입니다. 340이면 가장 긴 쌍이 378글자입니다.
- `PRIVATE`: 처음 만들 때는 `True`로 올려 확인한 뒤 공개로 바꾸는 것을 권합니다. 이미 공개한 저장소를 다시 올릴 때는 `False`로 둡니다.


In [ ]:
SKIP = 1_000_000          # 앞 100만 편은 사전학습용으로 남겨 둠
N_STORIES = 150_000       # 살펴볼 이야기 수
MAX_LEN = 340             # 이야기 최대 글자 수
VAL_RATIO = 0.05          # 검증용 비율
SEED = 42

HF_REPO = "JosephLee/tiny-ko-stories-sft"
PRIVATE = False           # 이미 공개한 저장소를 다시 올리므로


## 2. 이야기 불러오기

스트리밍으로 `SKIP`편을 건너뛴 뒤 `N_STORIES`편을 읽습니다. 건너뛰는 데 몇 분 걸릴 수 있습니다.


In [ ]:
from datasets import load_dataset

stream = load_dataset("psymon/Tiny-Ko-Stories", split="train", streaming=True)
stories = [row["text"] for row in stream.skip(SKIP).take(N_STORIES)]
print("불러온 이야기:", len(stories))
print(stories[0][:200])


## 3. (선택) 이름 후보 확인

이야기 첫머리에 자주 나오는 "○○는/○○가"를 세어 봅니다. 4번 셀의 `NAMES` 목록을 다듬을 때 참고하세요.
꾸밈말과 헷갈리는 이름(예: "반짝이는 언덕"의 반짝이)은 목록에서 빼는 것이 안전합니다.


In [ ]:
import re
from collections import Counter

starts = Counter()
for s in stories[:20000]:
    m = re.match(r"\s*([가-힣]{2,4})(는|가)\s", s)
    if m:
        starts[m.group(1)] += 1
print(starts.most_common(40))


## 4. 특징 찾기 규칙

- 이야기 **앞 두 문장**에서 이름·동물·때를 찾습니다. 주인공은 대개 여기서 소개되기 때문입니다.
- "토끼 보리"처럼 동물과 이름이 붙어 있을 때만 하나로 묶습니다. 따로 나오면 둘의 관계를 알 수 없으므로 묶지 않습니다.
- 요청 문장은 찾은 특징으로 만들 수 있는 틀 중에서 무작위로 고릅니다. 이야기에 실제로 있는 내용만 요청에 들어갑니다.


In [ ]:
import random
import re

# ── 1. 특징 사전 ─────────────────────────────────────────
# 이름: (본문에서 찾을 형태, 요청 문장에 쓸 형태)
NAMES = [
    ("하윤", "하윤이"), ("서준", "서준이"), ("도윤", "도윤이"), ("민준", "민준이"),
    ("서윤", "서윤이"), ("하준", "하준이"), ("복실이", "복실이"), ("동글이", "동글이"), ("지우", "지우"), ("지호", "지호"), ("수아", "수아"),
    ("민서", "민서"), ("두리", "두리"), ("보리", "보리"), ("토리", "토리"),
    ("콩이", "콩이"), ("뭉치", "뭉치"),
]

ANIMALS = ["강아지", "고양이", "토끼", "사자", "다람쥐", "오리", "나비",
           "꿀벌", "개미", "거북", "곰", "물고기", "새"]

# 긴 것부터 검사해야 "숲속"이 "숲"보다 먼저 잡힘
PLACES = ["동물원", "도서관", "놀이터", "바닷가", "시냇물", "꽃밭", "숲속", "부엌",
          "시장", "교실", "학교", "마당", "공원", "거실", "연못", "개울", "언덕",
          "들판", "정원", "가게", "강가", "풀밭", "마을", "바다", "숲"]

TIMES = {"봄날": "봄날", "여름날": "여름날", "가을날": "가을날", "겨울날": "겨울날",
         "아침": "아침", "점심": "점심시간", "저녁": "저녁", "밤": "밤"}

# "반짝이"처럼 꾸밈말과 헷갈리는 이름은 목록에서 뺐음
NAME_AFTER = r"(?=[이는은가와과도의를을에께랑\s,.!?]|$)"

# 동물 이름 뒤에 올 수 있는 글자 (다른 낱말의 일부로 잘못 잡히는 것 막기)
AFTER_ANIMAL = r"(?=[\s가이는은를을와과의도에]|$)"


# ── 2. 조사 붙이기 ───────────────────────────────────────
def has_batchim(word):
    code = ord(word[-1]) - 0xAC00
    return 0 <= code <= 11171 and code % 28 != 0

def josa(word, with_b, without_b):
    return word + (with_b if has_batchim(word) else without_b)


# ── 3. 이야기에서 특징 찾기 ──────────────────────────────
def first_sentences(story, n=2):
    parts = re.split(r"(?<=[.!?])\s+", story.strip())
    return " ".join(parts[:n])

def find_name(head):
    found = []
    for pattern, display in NAMES:
        m = re.search(r"(?:^|[\s\"'])" + pattern + NAME_AFTER, head)
        if m:
            found.append((m.start(), display))
    return min(found)[1] if found else None

def find_animal(head):
    found = []
    for a in ANIMALS:
        m = re.search(r"(?:^|\s)(?:아기 |꼬마 |작은 )?" + a + AFTER_ANIMAL, head)
        if m:
            found.append((m.start(), a))
    return min(found)[1] if found else None

def find_place(story):
    found = []
    for p in PLACES:
        i = story.find(p)
        if story[i + len(p):i + len(p) + 2] == "처럼":   # "언덕처럼" 같은 비유 제외
            continue
        if i >= 0 and not any(p in q and p != q and story.find(q) == i for q in PLACES):
            found.append((i, p))
    return min(found)[1] if found else None

def find_time(head):
    found = [(head.find(k), v) for k, v in TIMES.items() if head.find(k) >= 0]
    return min(found)[1] if found else None

def extract(story):
    head = first_sentences(story, 2)
    name, animal = find_name(head), find_animal(head)
    # "토끼 보리"처럼 동물과 이름이 붙어 있을 때만 하나로 묶음
    hero = None
    if name and animal and re.search(animal + r"\s" + re.escape(name[:2]), head):
        hero = f"{animal} {name}"
    return {"name": name, "animal": animal, "hero": hero,
            "place": find_place(head) or find_place(story[:150]),
            "time": find_time(head)}


# ── 4. 요청 문장 만들기 ──────────────────────────────────
ENDINGS = ["이야기를 들려줘", "이야기를 써 줘", "이야기를 만들어 줘", "이야기 해 줘"]

def make_request(f, rng):
    who = f["hero"] or f["name"]
    options = []
    if who and f["place"]:
        options.append(f"{f['place']}에서 {josa(who, '이가', '가')} 나오는 ")
        options.append(f"{josa(who, '이가', '가')} {f['place']}에서 겪은 ")
    if who:
        options.append(f"{josa(who, '이가', '가')} 나오는 ")
    if f["animal"] and not f["hero"]:
        options.append(f"{josa(f['animal'], '이', '가')} 나오는 ")
        if f["place"]:
            options.append(f"{f['place']}에서 {josa(f['animal'], '이', '가')} 나오는 ")
    if f["time"] and f["place"]:
        options.append(f"{f['time']} {f['place']}에서 일어난 ")
    if f["place"] and not options:
        options.append(f"{f['place']}에서 일어난 ")
    if not options:
        return None
    return rng.choice(options) + rng.choice(ENDINGS)


def build_pairs(stories, max_len=300, seed=42):
    rng = random.Random(seed)
    pairs = []
    for s in stories:
        s = s.strip()
        if len(s) > max_len:
            continue
        req = make_request(extract(s), rng)
        if req:
            pairs.append({"request": req, "story": s})
    return pairs


## 5. 쌍 만들고 점검하기

무작위 샘플을 직접 읽어 보는 것이 가장 중요한 점검입니다. 요청과 이야기가 어긋나는 예가 보이면 4번 셀의 목록을 고치고 다시 실행하세요.


In [ ]:
import random

pairs = build_pairs(stories, max_len=MAX_LEN, seed=SEED)
short = sum(len(s.strip()) <= MAX_LEN for s in stories)
print(f"길이 조건 통과: {short:,}편 → 요청 생성 성공: {len(pairs):,}쌍 ({len(pairs)/max(short,1):.1%})")

# 요청 틀 분포
kinds = Counter(re.sub(r"(이야기.*)$", "", p["request"]).split()[-1] for p in pairs)
print("요청 틀(끝말 기준):", kinds.most_common())

for p in random.Random(0).sample(pairs, 10):
    print("\n요청:", p["request"])
    print("이야기:", p["story"][:120], "...")


## 6. 글자 점검

책의 모델은 **글자 하나가 토큰 하나**입니다. 요청 문장이나 형식 표시(`요청:`, `이야기:`, 끝 표시 `§`)에 이야기 본문에 없는 글자가 있으면,
사전학습 단계의 글자 사전에 미리 넣어 두어야 합니다. 여기 출력된 글자를 사전학습 노트북의 사전에 추가하세요.


In [ ]:
story_chars = set("".join(p["story"] for p in pairs))
extra = set("".join(p["request"] for p in pairs)) | set("요청: 이야기:\n§")
missing = sorted(extra - story_chars)
print("이야기에 없는 글자:", missing if missing else "없음")


## 7. 나누고 저장하기

업로드 전에 Colab에도 JSONL로 저장해 둡니다.

In [ ]:
from datasets import Dataset, DatasetDict

random.Random(SEED).shuffle(pairs)
n_val = int(len(pairs) * VAL_RATIO)
ds = DatasetDict({
    "train": Dataset.from_list(pairs[n_val:]),
    "validation": Dataset.from_list(pairs[:n_val]),
})
print(ds)
ds["train"].to_json("sft_train.jsonl", force_ascii=False)
ds["validation"].to_json("sft_validation.jsonl", force_ascii=False)


## 8. 데이터셋 카드

CC BY 4.0의 출처 표시(원 데이터 이름·작성자·링크·라이선스)와 **변경 사항**을 적습니다.


In [ ]:
card = f"""---
license: cc-by-4.0
language:
- ko
task_categories:
- text-generation
tags:
- korean
- children-stories
- instruction-tuning
- education
pretty_name: Tiny-Ko-Stories SFT
---

# Tiny-Ko-Stories SFT (요청 → 이야기)

『모두의 LLM with 파이썬』 실습용 지시 미세조정(SFT) 데이터입니다.

| 필드 | 내용 |
|---|---|
| `request` | 이야기를 요청하는 짧은 문장 |
| `story` | 요청에 해당하는 어린이 이야기 |

- train {len(ds["train"]):,}쌍 / validation {len(ds["validation"]):,}쌍
- 이야기 길이 {MAX_LEN}자 이하

## 출처와 변경 사항

이 데이터는 psymon의 [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories)
([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/))를 가공한 것입니다.

- 원본의 {SKIP+1:,}번째 이야기부터 {N_STORIES:,}편을 사용했습니다.
- {MAX_LEN}자를 넘는 이야기는 제외했습니다.
- 이야기 앞부분에서 등장인물 이름, 동물, 장소, 때를 규칙으로 찾아 요청 문장을 자동 생성했습니다.
- 이야기 본문은 바꾸지 않았습니다.

원본은 여러 AI 모델로 생성된 합성 데이터이며, 원본 카드의 권장·비권장 용도와 한계가 이 데이터에도 그대로 적용됩니다.
사실 질의응답 학습이나 사람 검수 없는 어린이 대상 서비스에는 적합하지 않습니다.

## 라이선스

CC BY 4.0. 사용할 때 이 데이터셋과 원본 Tiny-Ko-Stories의 출처를 함께 표기해 주세요.
"""
print(card[:400])


## 9. 허깅페이스에 올리기

쓰기(write) 권한이 있는 토큰으로 로그인합니다. `PRIVATE=True`로 먼저 올려 웹에서 확인한 뒤, 저장소 설정에서 공개로 바꾸세요.


In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()


In [ ]:
ds.push_to_hub(HF_REPO, private=PRIVATE)
HfApi().upload_file(path_or_fileobj=card.encode("utf-8"), path_in_repo="README.md",
                    repo_id=HF_REPO, repo_type="dataset")
print("완료:", f"https://huggingface.co/datasets/{HF_REPO}")


## 10. 불러오기 확인

책의 노트북에서 쓸 코드와 같은 방식으로 불러와 봅니다. 비공개 상태에서는 로그인한 계정에서만 불러올 수 있습니다.


In [ ]:
check = load_dataset(HF_REPO)
print(check)
print(check["train"][0])
